<a href="https://colab.research.google.com/github/rafi1624/Machine-Learning_18_Rafi-Abyantara/blob/main/JS04/JS04-Tugas%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Membaca dataset
df = pd.read_csv('/content/insurance.csv')

# Menampilkan 5 data teratas
display(df.head())

# Menampilkan informasi dataset
print(df.info())

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB
None


In [ ]:
# One-Hot Encoding untuk kolom kategorikal
df_encoded = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True)

# Menentukan Fitur (X) dan Target (y)
X = df_encoded.drop(columns=['charges'])
y = df_encoded['charges']

print("Fitur yang digunakan:", list(X.columns))
print("Target:", y.name)
display(X.head())

Fitur yang digunakan: ['age', 'bmi', 'children', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']
Target: charges


,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27.900,0,False,True,False,False,True
1,18,33.770,1,True,False,False,True,False
2,28,33.000,3,True,False,False,True,False
3,33,22.705,0,True,False,True,False,False
4,32,28.880,0,True,False,True,False,False


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Membagi data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Inisialisasi Scaler
scaler_X = StandardScaler()

# Scaling fitur
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

print(f"Jumlah data latih: {X_train.shape[0]}")
print(f"Jumlah data uji: {X_test.shape[0]}")

Jumlah data latih: 1070
Jumlah data uji: 268


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Inisialisasi dan melatih model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Prediksi
y_pred_lr = lr_model.predict(X_test_scaled)

# Evaluasi
mse_lr = mean_squared_error(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("=== Evaluasi Multiple Linear Regression ===")
print(f"R-squared (R2) : {r2_lr:.4f}")
print(f"MSE            : {mse_lr:.4f}")
print(f"MAE            : {mae_lr:.4f}")

=== Evaluasi Multiple Linear Regression ===
R-squared (R2) : 0.7836
MSE            : 33596915.8514
MAE            : 4181.1945


In [6]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

# SVR membutuhkan target scaling untuk hasil maksimal
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()

# Inisialisasi model SVR dasar
svr = SVR()

# Parameter yang akan diuji saat Grid Search
param_grid = {
    'kernel': ['rbf', 'linear'],
    'C': [1, 10, 100, 1000],
    'gamma': ['scale', 'auto', 0.01, 0.1]
}

# Hyperparameter Tuning menggunakan GridSearchCV
grid_search = GridSearchCV(svr, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train_scaled)

# Model SVR terbaik
best_svr = grid_search.best_estimator_
print("Hyperparameter terbaik untuk SVR:", grid_search.best_params_)

# Prediksi (dalam skala ter-scaling)
y_pred_svr_scaled = best_svr.predict(X_test_scaled)

# Mengembalikan hasil prediksi ke skala asli (Inverse Transform)
y_pred_svr = scaler_y.inverse_transform(y_pred_svr_scaled.reshape(-1, 1)).ravel()

# Evaluasi SVR
mse_svr = mean_squared_error(y_test, y_pred_svr)
mae_svr = mean_absolute_error(y_test, y_pred_svr)
r2_svr = r2_score(y_test, y_pred_svr)

print("\n=== Evaluasi Support Vector Regression (SVR) ===")
print(f"R-squared (R2) : {r2_svr:.4f}")
print(f"MSE            : {mse_svr:.4f}")
print(f"MAE            : {mae_svr:.4f}")

Hyperparameter terbaik untuk SVR: {'C': 100, 'gamma': 0.01, 'kernel': 'rbf'}

=== Evaluasi Support Vector Regression (SVR) ===
R-squared (R2) : 0.8690
MSE            : 20345260.1086
MAE            : 2417.0261


### 6. Perbandingan Hasil Evaluasi Model

In [7]:
comparison_df = pd.DataFrame({
    'Metric': ['R-squared (R2)', 'MSE', 'MAE'],
    'Linear Regression': [r2_lr, mse_lr, mae_lr],
    'SVR (Tuned)': [r2_svr, mse_svr, mae_svr]
})

display(comparison_df)

,Metric,Linear Regression,SVR (Tuned)
0,R-squared (R2),7.835930e-01,8.689506e-01
1,MSE,3.359692e+07,2.034526e+07
2,MAE,4.181194e+03,2.417026e+03
